In [1]:
from datasets import load_from_disk

ds = load_from_disk(f'/home/husain/data/ArabicVoicesClean_v4')

ds

/home/husain/work/asr_leaderboard/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['audio_filepath', 'text', 'speaker', 'duration', 'session_id', 'start', 'end', 'word_count', 'audio'],
        num_rows: 22843
    })
})

In [2]:
filtered = ds.filter(lambda x: x['audio_filepath'] == 'clips/06ovyUh2iWU_0062.wav')

filtered

DatasetDict({
    train: Dataset({
        features: ['audio_filepath', 'text', 'speaker', 'duration', 'session_id', 'start', 'end', 'word_count', 'audio'],
        num_rows: 1
    })
})

In [24]:
from datasets import Audio

In [27]:
ds = ds.cast_column('audio', Audio(decode=False))

In [ ]:
audio = ds['train'][0]['audio']

In [31]:
audio['bytes']

b'RIFF\xa6\xf4\x03\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80>\x00\x00\x00}\x00\x00\x02\x00\x10\x00LIST\x1a\x00\x00\x00INFOISFT\x0e\x00\x00\x00Lavf58.76.100\x00data`\xf4\x03\x00:\xfcZ\xfc\x16\xfc]\xfcW\xfb]\xfc\xfe\xfbL\xfc\x1d\xfd=\xfdn\xfdn\xfcM\xfe\xac\xfd\x06\xfe\x9c\xff\xf3\xfd@\x00\'\xff\x9c\xffw\x00\x1d\x00\xfc\x00M\x00V\x01\xc3\x01N\x01\xee\x00\xa7\x01"\x02\xf3\x02~\x01\xc0\x00\xdf\x02R\x02\xa2\x01\xb8\x02\xb9\x02\x0c\x02M\x02~\x02\xcd\x06^\t\xe4\x05\xf0\x01\xac\x03\xda\x0bR\x0e\xc0\x0cd\ne\t\xa2\rB\x10\xb2\x0e\x19\x0e\x16\x0e\xac\x0f\x9f\x10\xed\x0fM\r!\x0c)\x0c\xe3\x0c\x97\r7\x0b\x03\x07\x12\x04{\x04\xca\x04#\x03j\x00O\xfd\x96\xfb\xe4\xfa\xd8\xf95\xf8A\xf7\xff\xf4Z\xf5\xb2\xf4;\xf4K\xf3+\xf2\xc9\xf2Y\xf4\xc5\xf4\xbc\xf5{\xf4]\xf5\x80\xf6\xb5\xf7\xc9\xf8\x06\xfa\xad\xfa\xc8\xfbw\xfb\xa8\xfc\xcc\xfdA\xfe\xb5\xfeY\xfe\x92\xfe\x9c\xff>\xffD\xfe\xd7\xfd\x03\xfe<\xfe)\xfeV\xfd\xe3\xfb0\xfa]\xfb\xee\xfa\xce\xfaH\xfaH\xf9=\xf9\xca\xf8\xc0\xf8&\xf9\x0c\xfa\xd9\xfa:\xfa\x1e\xfbm\x

In [18]:
import sounddevice as sd

sd.play(audio['array'], audio['sampling_rate'])
sd.wait()

KeyboardInterrupt: 

طبعا هو مش عايز يديني رقم ولا هتعرف تجيب الرقم بس أنا كنت هكلمه أقوله طب إنت ليه تديله إنت بعتله الحاجة دي هي دي أصلاً ما تنفعوش

طبعًا هو ما يبغى يعطيني رقم ولا بيعرف يجيب لي الرقم لكن كنت بكلمه وبقول له طيب انت ليه تعطيه انت بعت له الحاجة ذي، هي ذي اصلًا ما تنفعه


In [5]:
filtered['train'][0]['text']

'طبعًا هو ما يبغى يعطيني رقم ولا بيعرف يجيب لي الرقم لكن كنت بكلمه وبقول له طيب انت ليه تعطيه انت بعت له الحاجة ذي، هي ذي اصلًا ما تنفعه'

In [11]:
import io
import soundfile as sf
from google import genai
from google.genai import types

client = genai.Client()

audio_array = audio['array']
sr = audio['sampling_rate']

buffer = io.BytesIO()
sf.write(buffer, audio_array, sr, format='WAV')
audio_bytes = buffer.getvalue()

response = client.models.generate_content(
    model='gemini-3-flash-preview',
    contents=[
        'Transcribe this audio exactly. It is in arabic. return JSON with key "text".',
        types.Part.from_bytes(data=audio_bytes, mime_type='audio/wav')
    ]
)

print(response.text)

```json
{
  "text": "طبعا هو مش عايز يديني رقم ولا هتعرف تجيب الرقم بس أنا كنت هكلمه أقوله طب إنت ليه تديله إنت بعتله الحاجة دي هي دي أصلاً ما تنفعوش"
}
```


In [ ]:
print(1)

In [1]:
from pathlib import Path
import json
import os
from eval import normalize_arabic_text

output_path = Path(f'gemini_labels.jsonl')
whisper_labels = Path(os.path.join('outputs/arabicvoices_labels','ArabicVoicesClean_v4.txt'))
finetuned_whisper_labels = Path(os.path.join('outputs/arabicvoices_labels_mymodelv2','ArabicVoicesClean_v4.txt'))

gemini_preds = []
whisper_preds = []
gt_labels =[]
finetuned_preds = []

with output_path.open('r', encoding='utf-8') as f:
    with whisper_labels.open('r', encoding='utf-8') as f1:
        with finetuned_whisper_labels.open('r', encoding='utf-8') as f2:
            i = 0
            recordsf1 = []
            recordsf2 = []
            for line in f1:
                recordsf1.append(json.loads(line)['pred_text'])

            for line in f2:
                recordsf2.append(json.loads(line)['pred_text'])

            for line in f:
                record = json.loads(line)

                gt_label = normalize_arabic_text(record['original_transcript'])
                gemini_label = normalize_arabic_text(record['gemini_transcript'])
                whisper_label = normalize_arabic_text(recordsf1[i])
                finetuned_label = normalize_arabic_text(recordsf2[i])

                gt_labels.append(gt_label)
                gemini_preds.append(gemini_label)
                whisper_preds.append(whisper_label)
                finetuned_preds.append(finetuned_label)

                print(f'original: {gt_label}')
                print(f'gemini: {gemini_label}')
                print(f'whisper: {whisper_label}')
                print(f'finetuned: {finetuned_label}')

                print('')
                print('')
                i+=1


[NeMo W 2026-03-11 11:13:56 nemo_logging:405] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-03-11 11:13:57 nemo_logging:405] c:\Users\husain_althagafi\work\leaderboard_asr\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
      warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
    


original: يعني يعني انا دوس على زرار الاساس ينزل ادوس على زرار ايه الحلاوه دي يعني انت مش متخيل ان في كده في الدنيا ايوه انا عشان
gemini: يعني يعني انا ادوس على زرار الازاز ينزل ادوس على زرار الازاز ايه الحلاوة دي يعني انت مش متخيل ان فيه كده في الدنيا ايوه انا عشان الف الدركسيون مش محتاج ابقى بلعب جيم
whisper: يعني انا ادوس على زرار الاستاذ ينزل ادوس على زرار الحلاوة دي يعني انت مش متخيل ان فيه كده في الدنيا انا عشان اليف الدريكسيون مش محتاج ابقى بلعب جيم
finetuned: يعني انا ادوس على زرار الاستاذ ينزل ادوس على زرار الحلاوه دي يعني انت مش متخيل ان فيه كده في الدنيا ايوه انا عشان الف الدريكسيون مش محتاجه ابقى بلعب جيم


original: رفاهيه صح مع الوقت وقت تحولت لحاجه اساسيه يعني ايه اساسيه تاني لو مش موجوده
gemini: رفاهية صح مع الوقت تحولت لحاجة اساسية يعني ايه اساسية تاني لو مش موجودة في السلعة او في الخدمة انا هزعل طبعا
whisper: رفاهية صح مع الوقت تحولت لحاجة اساسية يعني ايه اساسية تاني لو مش موجودة في السلعة او في الخدمة انا هزعل طبعا
finetuned: رفاهيه صح مع الوقت تحولت لحاجه اساسيه يعن

In [12]:
from nemo.collections.asr.metrics.wer import word_error_rate

print(f'WER between gt and gemini: {word_error_rate(gemini_preds, gt_labels)} CER between gt and gemini: {word_error_rate(gemini_preds, gt_labels, use_cer=True)}')
print(f'WER between gt and whisper: {word_error_rate(whisper_preds, gt_labels)} CER between gt and whisper: {word_error_rate(whisper_preds, gt_labels, use_cer=True)}')
print(f'WER between gemini and whisper: {word_error_rate(gemini_preds, whisper_preds)} CER between gemini and whisper: {word_error_rate(gemini_preds, whisper_preds, use_cer=True)}')
print(f'WER between gt and finetuned: {word_error_rate(gt_labels, finetuned_preds)} CER between gt and finetuned: {word_error_rate(gt_labels, finetuned_preds, use_cer=True)}')


WER between gt and gemini: 0.5714285714285714 CER between gt and gemini: 0.39956331877729256
WER between gt and whisper: 0.5608465608465608 CER between gt and whisper: 0.37554585152838427
WER between gemini and whisper: 0.2826086956521739 CER between gemini and whisper: 0.15861440291704648
WER between gt and finetuned: 0.425531914893617 CER between gt and finetuned: 0.31312217194570136
